<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/05_generativos/51_gan.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# Redes generativas adversarias

**Pregunta guía:** ¿Cómo aprende un generador mediante un adversario?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


**Requiere PyTorch; GPU opcional.** El juego clásico es
$\min_G\max_D\;E_{x\sim p_d}\log D(x)+E_z\log(1-D(G(z)))$.
Para el generador usamos la pérdida no saturante $-\log D(G(z))$.
Entrenaremos sobre dígitos 8×8: es un laboratorio de dinámica, no un
generador de alta fidelidad.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.datasets import load_digits
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEMILLA=42; torch.manual_seed(SEMILLA)
dispositivo=torch.device("cuda" if torch.cuda.is_available() else "cpu")
X=torch.tensor(load_digits().data/16.0,dtype=torch.float32)
loader=DataLoader(TensorDataset(X),batch_size=128,shuffle=True,generator=torch.Generator().manual_seed(SEMILLA),drop_last=True)
G=nn.Sequential(nn.Linear(16,64),nn.LeakyReLU(.2),nn.Linear(64,128),nn.LeakyReLU(.2),nn.Linear(128,64),nn.Sigmoid()).to(dispositivo)
D=nn.Sequential(nn.Linear(64,128),nn.LeakyReLU(.2),nn.Dropout(.2),nn.Linear(128,64),nn.LeakyReLU(.2),nn.Linear(64,1)).to(dispositivo)
optG=torch.optim.Adam(G.parameters(),lr=2e-4,betas=(.5,.999)); optD=torch.optim.Adam(D.parameters(),lr=2e-4,betas=(.5,.999))
bce=nn.BCEWithLogitsLoss(); historia=[]


In [ ]:
for época in range(120):
    for (real,) in loader:
        real=real.to(dispositivo); n=len(real)
        z=torch.randn(n,16,device=dispositivo); falso=G(z)
        optD.zero_grad()
        lossD=bce(D(real),torch.ones(n,1,device=dispositivo))+bce(D(falso.detach()),torch.zeros(n,1,device=dispositivo))
        lossD.backward(); optD.step()
        z=torch.randn(n,16,device=dispositivo)
        optG.zero_grad(); lossG=bce(D(G(z)),torch.ones(n,1,device=dispositivo)); lossG.backward(); optG.step()
    historia.append((lossD.item(),lossG.item()))
    if época%30==0: print(época, historia[-1])


In [ ]:
hist=np.asarray(historia)
plt.plot(hist[:,0],label="D"); plt.plot(hist[:,1],label="G"); plt.legend(); plt.xlabel("época"); plt.show()
with torch.no_grad(): muestras=G(torch.randn(20,16,device=dispositivo)).cpu().reshape(-1,8,8)
fig,axes=plt.subplots(4,5,figsize=(7,6))
for ax,img in zip(axes.flat,muestras): ax.imshow(img,cmap="gray"); ax.axis("off")
plt.show()


Las pérdidas no son una métrica suficiente: un generador puede colapsar
a pocos modos. **Ejercicios:** guarde muestras cada 10 épocas; mida
diversidad entre pares; alterne más pasos de D; compare ruido latente;
investigue WGAN conceptualmente y explique qué cambia en la distancia.
